## ② 프롬프트 3종 × 모델 2종 비교

### 준비
- [ ] 상용 API 모델 하나 선정
- [ ] 오픈 LLM 선정: 포르투갈어가 되는 소형 instruct 모델
- [ ] 오픈 LLM을 vLLM/Ollama 등으로 Docker에서 서빙. OpenAI 호환 엔드포인트를 쓰면 두 모델을 같은 코드로 호출할 수 있다.
- [ ] 다국어 임베딩 모델로 예시 풀을 벡터화해 pgvector에 적재

### 실험 설계

세 프롬프트 모두 예시 수 k, 출력 형식(JSON), temperature=0을 동일하게 맞춘다.

| 프롬프트 | 내용 |
|---|---|
| zero-shot | 카테고리 정의만 제공 |
| 고정 few-shot | 풀에서 고른 대표 예시 k개를 고정 |
| RAG few-shot | 리뷰마다 유사도 top-k 예시를 검색 |
| (권장) 랜덤 few-shot | 리뷰마다 풀에서 랜덤 k개 — 대조군 |

**효과 분리**
- 예시 효과 = 고정 few-shot − zero-shot
- 검색 효과 = RAG − 고정 few-shot
- RAG와 고정 few-shot은 "리뷰마다 예시가 바뀐다"는 점과 "비슷한 예시다"라는 점이 동시에 다르다. 랜덤 few-shot 대조군과 비교해야 순수한 유사도 효과를 분리할 수 있다.

### 지표
- **정확도:** macro-F1, 클래스별 F1, 혼동행렬, JSON 파싱 실패율
- **비용:** API는 토큰 × 단가, 로컬은 GPU 시간 환산
- **속도:** 지연시간 p50/p95, 처리량
- 평가셋이 작으므로 부트스트랩 신뢰구간을 함께 보고한다.

### 마무리
- [ ] 설정 파일로 돌아가는 실험 스크립트 작성, 결과표 CSV 저장
- [ ] 최종 조합을 골라 대상 전체(약 1만 건) 분류

**산출물:** 비교 결과표, 선택 근거, 전체 예측 결과

In [ ]:
# !pip install ollama

  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 452.9 kB/s  0:00:06 eta 0:00:01
Using cached annotated_types-0.8.0-py3-none-any.whl (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [ollama]10/11 [ollama]c]


In [16]:
# ===== 라이브러리 =====
import json
import time
import re

import ollama
import polars as pl

from pathlib import Path

# ===== 경로 설정 =====
DATA_DIR = "../data"
processed_dir = Path(DATA_DIR) / "processed"

# ollama 연결 및 분류 테스트

In [ ]:
# ── 설정: 모델별 옵션과 라벨 (OUTPUT_SCHEMA는 삭제) ──
MODEL_CONFIGS = {
    "qwen3.5:4b": {"think": False},
    "gemma4:e2b-it-qat": {},
}

LABELS = ["배송 지연", "누락", "제품 오배송", "제품 불량", "위조·비정품", "제품 불만족", "기타"]

# 연결 확인용 임시 프롬프트 (정식 프롬프트는 다음 단계에서 설계)
TEST_PROMPT = (
    "브라질 이커머스의 포르투갈어 불만 리뷰를 읽고, 불만의 주 원인을 "
    f"다음 라벨 중 하나로 분류하라: {', '.join(LABELS)}\n"
    '반드시 {"label": "라벨"} 형식의 JSON 한 줄로만 답하라.'
)

In [4]:
# ── 테스트용 리뷰 1건 불러오기 ──
# 경로는 실제 위치에 맞게 수정
pool_df = pl.read_csv(processed_dir / "pool.csv")
# utf-8-sig로 저장돼 첫 컬럼명에 BOM이 붙을 수 있어 제거
pool_df = pool_df.rename(lambda col_name: col_name.lstrip("\ufeff"))

sample_row = pool_df.row(0, named=True)
print(sample_row["label"], "|", sample_row["review_text"])

배송 지연 | Ainda não recebi o meu pedido, estava programado para o dia 13/12 e até agora nada, hoje já é dia 17/12


In [11]:
# ── 응답 파싱: JSON 부분만 추출 → 라벨 검증 ──
# 모델이 ```json 코드블록이나 설명 문장을 붙이는 경우를 대비해 첫 { ... } 만 꺼냄
def parse_label(raw_text):
    json_match = re.search(r"\{.*?\}", raw_text, re.DOTALL)
    if json_match is None:
        return None
    try:
        label = json.loads(json_match.group()).get("label")
    except json.JSONDecodeError:
        return None
    # 7개 라벨 중 하나가 아니면 실패로 처리
    return label if label in LABELS else None


# ── 분류 요청: 결과를 dict 하나로 반환 ──
def classify_review(model_name, system_prompt, review_text):
    start_time = time.perf_counter()
    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": review_text},
        ],
        options={"temperature": 0},
        **MODEL_CONFIGS[model_name],
    )
    elapsed_sec = time.perf_counter() - start_time

    raw_output = response.message.content
    predicted_label = parse_label(raw_output)
    return {
        "model": model_name,
        "pred_label": predicted_label,
        "parse_ok": predicted_label is not None,  # 파싱 실패율 계산용
        "raw_output": raw_output,                 # 실패 원인 확인용 원문
        "elapsed_sec": round(elapsed_sec, 2),
        "input_tokens": response.prompt_eval_count,
        "output_tokens": response.eval_count,
    }

In [15]:
# ── 테스트: 모델 하나씩, 워밍업 1회 후 측정 ──
test_results = []
for model_name in MODEL_CONFIGS:
    classify_review(model_name, TEST_PROMPT, sample_row["review_text"])  # 워밍업 (로딩 시간 제외)
    test_results.append(classify_review(model_name, TEST_PROMPT, sample_row["review_text"]))

test_df = pl.DataFrame(test_results).with_columns(pl.lit(sample_row["label"]).alias("gold_label"))
with pl.Config(fmt_str_lengths=100):
    print(test_df)

shape: (2, 8)
┌────────────┬────────────┬──────────┬────────────┬────────────┬───────────┬───────────┬───────────┐
│ model      ┆ pred_label ┆ parse_ok ┆ raw_output ┆ elapsed_se ┆ input_tok ┆ output_to ┆ gold_labe │
│ ---        ┆ ---        ┆ ---      ┆ ---        ┆ c          ┆ ens       ┆ kens      ┆ l         │
│ str        ┆ str        ┆ bool     ┆ str        ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│            ┆            ┆          ┆            ┆ f64        ┆ i64       ┆ i64       ┆ str       │
╞════════════╪════════════╪══════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╡
│ qwen3.5:4b ┆ 배송 지연  ┆ true     ┆ {"label":  ┆ 0.47       ┆ 129       ┆ 7         ┆ 배송 지연 │
│            ┆            ┆          ┆ "배송      ┆            ┆           ┆           ┆           │
│            ┆            ┆          ┆ 지연"}     ┆            ┆           ┆           ┆           │
│ gemma4:e4b ┆ 배송 지연  ┆ true     ┆ {"label":  ┆ 0.78       ┆ 135       ┆ 10        ┆ 배송 지

# 분류 프롬프트 스모크 테스트

In [27]:
# ── 설정: 모델별 옵션과 라벨 (OUTPUT_SCHEMA는 삭제) ──
MODEL_CONFIGS = {
    "qwen3.5:4b": {"think": False},
    "gemma4:e2b-it-qat": {},
}

LABELS = ["배송 지연", "누락", "제품 오배송", "제품 불량", "위조·비정품", "제품 불만족", "기타"]

In [28]:
# ── 테스트용 리뷰 1건 불러오기 ──
# 경로는 실제 위치에 맞게 수정
pool_df = pl.read_csv(processed_dir / "pool.csv")
# utf-8-sig로 저장돼 첫 컬럼명에 BOM이 붙을 수 있어 제거
pool_df = pool_df.rename(lambda col_name: col_name.lstrip("\ufeff"))

sample_row = pool_df.row(0, named=True)
print(sample_row["label"], "|", sample_row["review_text"])

배송 지연 | Ainda não recebi o meu pedido, estava programado para o dia 13/12 e até agora nada, hoje já é dia 17/12


In [29]:
# ── 라벨 코드 ↔ 한글 매핑 (모델은 코드로 답하고, 평가할 때 한글로 되돌림) ──
LABEL_CODE_TO_KO = {
    "delivery_delay": "배송 지연",
    "missing_item": "누락",
    "wrong_item": "제품 오배송",
    "defective": "제품 불량",
    "counterfeit": "위조·비정품",
    "dissatisfied": "제품 불만족",
    "other": "기타",
}

# ── 공통 기본 프롬프트 (zero-shot은 이것만, few-shot은 여기에 예시를 덧붙임) ──
BASE_PROMPT = """You classify negative customer reviews (in Brazilian Portuguese) from a Brazilian e-commerce marketplace by the main cause of the complaint. Use only the review text.

Labels:
- delivery_delay: The product arrived late or has not arrived. Includes lost packages, marked as delivered but not received, delivered to a wrong address or city, and delays in exchange or re-shipment.
- wrong_item: A different product was sent, or an objective specification differs from the listing (model, brand, color, size, capacity, material, voltage, pattern).
- defective: The received product has an objective defect: does not work, broken, dented, scratched, looks used, poor finish or assembly, expired or spoiled.
- missing_item: Part of the ordered quantity or a component did not arrive (e.g., bought 2 and received 1; missing parts or filter; a box the listing said was included).
- counterfeit: Sold as a genuine brand product, but a fake, replica, generic, or unbranded product arrived.
- dissatisfied: No objective defect, but the customer is subjectively unhappy with the product: quality, material, smell, performance, battery life, "not what I expected", unclear product description, close to expiration date, arrived without a box or poorly packed.
- other: None of the above, or the cause is unclear: vague complaints that do not mention the product ("bad", "don't recommend"), payment, cancellation, stock or customer-service issues, price or shipping cost, having to pick up at the post office, only the outer packaging damaged while the product is fine, praise despite a low score.

Rules:
1. If several causes appear, use this priority: wrong_item / defective / missing_item / counterfeit > dissatisfied > delivery_delay > other. Among the first four, choose the most specifically described; if equally specific, the one mentioned first.
2. Exception: if the main complaint is that an exchange or re-shipment for a product problem is taking too long, choose delivery_delay. This does not apply to refunds for products the customer simply did not like.
3. If only part of the order arrived, choose delivery_delay only when the text gives evidence that the rest is on the way (in transit, at the post office, arrived late, the promised date has passed, the seller said it was shipped separately). Otherwise choose missing_item, even if the customer says they are still waiting or hopes it is coming.
4. Compatibility: if the seller stated compatibility with the customer's device or car and it does not fit, choose wrong_item. If it is unclear whose mistake it is, choose other. Damage caused by the customer's misuse is other.
5. "Different from the photo/ad": choose wrong_item only if the review says what differs (color, size, model...) or says the wrong product was sent; otherwise dissatisfied. Calling it "a different product" only because of poor quality, "false advertising" without a specific difference, or cheap-knockoff slang ("paraguaio") without saying it was sold as genuine are all dissatisfied.
6. If the title and body contradict each other, follow the body.
7. Short delivery mentions ("still waiting", "where is my product?", "delivery problem") are delivery_delay. Waiting for something unspecified (e.g., "waiting for a reply") is other.

Answer with one line of JSON only: {"label": "<label>"}"""

In [30]:
# ── 응답 파싱: JSON 부분 추출 → 라벨 코드 검증 ──
def parse_label_code(raw_text):
    json_match = re.search(r"\{.*?\}", raw_text, re.DOTALL)
    if json_match is None:
        return None
    try:
        label_code = json.loads(json_match.group()).get("label")
    except json.JSONDecodeError:
        return None
    if not isinstance(label_code, str):
        return None
    # 대소문자·공백 차이만 정리하고, 7개 코드가 아니면 실패
    label_code = label_code.strip().lower()
    return label_code if label_code in LABEL_CODE_TO_KO else None


# ── 분류 요청: 결과를 dict 하나로 반환 ──
def classify_review(model_name, system_prompt, review_text):
    start_time = time.perf_counter()
    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": review_text},
        ],
        options={"temperature": 0},
        **MODEL_CONFIGS[model_name],
    )
    elapsed_sec = time.perf_counter() - start_time

    raw_output = response.message.content
    pred_code = parse_label_code(raw_output)
    return {
        "model": model_name,
        "pred_code": pred_code,
        "pred_label": LABEL_CODE_TO_KO.get(pred_code),  # 파싱 실패면 None
        "parse_ok": pred_code is not None,
        "raw_output": raw_output,
        "elapsed_sec": round(elapsed_sec, 2),
        "input_tokens": response.prompt_eval_count,
        "output_tokens": response.eval_count,
    }

In [ ]:
# ── 스모크 테스트: pool 20건을 모델 하나씩 zero-shot으로 분류 ──
smoke_df = pool_df.sample(n=20, seed=2026)

smoke_records = []
for model_name in MODEL_CONFIGS:
    if model_name != 'gemma4:e2b-it-qat': continue
    print(model_name)
    classify_review(model_name, BASE_PROMPT, smoke_df["review_text"][0])  # 워밍업
    for review_row in smoke_df.iter_rows(named=True):
        result = classify_review(model_name, BASE_PROMPT, review_row["review_text"])
        result.update(review_id=review_row["review_id"], gold_label=review_row["label"],
                      review_text=review_row["review_text"])
        smoke_records.append(result)

smoke_result_df = pl.DataFrame(smoke_records).with_columns(
    # 파싱 실패(None)는 오답으로 처리
    (pl.col("pred_label") == pl.col("gold_label")).fill_null(False).alias("is_correct")
)

In [33]:
# ── 모델별 요약: 정확도, 파싱 실패율, 지연시간, 입력 토큰 ──
smoke_summary_df = smoke_result_df.group_by("model").agg(
    pl.col("is_correct").mean().round(2).alias("accuracy"),
    (~pl.col("parse_ok")).mean().round(2).alias("parse_fail_rate"),
    pl.col("elapsed_sec").median().alias("p50_sec"),
    pl.col("input_tokens").mean().round(0).alias("avg_input_tokens"),
)
print(smoke_summary_df)

# ── 오답 확인: 정답·예측·리뷰 원문 (파싱 실패는 raw_output으로 원인 확인) ──
with pl.Config(fmt_str_lengths=150, tbl_rows=-1):
    print(
        smoke_result_df.filter(~pl.col("is_correct"))
        .select("model", "gold_label", "pred_label", "review_text", "raw_output")
    )

shape: (2, 5)
┌───────────────────┬──────────┬─────────────────┬─────────┬──────────────────┐
│ model             ┆ accuracy ┆ parse_fail_rate ┆ p50_sec ┆ avg_input_tokens │
│ ---               ┆ ---      ┆ ---             ┆ ---     ┆ ---              │
│ str               ┆ f64      ┆ f64             ┆ f64     ┆ f64              │
╞═══════════════════╪══════════╪═════════════════╪═════════╪══════════════════╡
│ gemma4:e2b-it-qat ┆ 0.9      ┆ 0.0             ┆ 14.335  ┆ 772.0            │
│ qwen3.5:4b        ┆ 0.95     ┆ 0.0             ┆ 0.905   ┆ 757.0            │
└───────────────────┴──────────┴─────────────────┴─────────┴──────────────────┘
shape: (3, 5)
┌───────────────────┬─────────────┬─────────────┬────────────────────────────────┬─────────────────┐
│ model             ┆ gold_label  ┆ pred_label  ┆ review_text                    ┆ raw_output      │
│ ---               ┆ ---         ┆ ---         ┆ ---                            ┆ ---             │
│ str               ┆ str    

# 임베딩

In [38]:
# ── 라이브러리 (numpy가 없으면 pip install numpy) ──
from collections import Counter

import numpy as np

# ── 설정 ──
EMBED_MODEL = "qwen3-embedding:0.6b"
EMBED_BATCH_SIZE = 32
EMBED_PATH = processed_dir / "embeddings_qwen3-0.6b.npz"
# eval 리뷰(검색하는 쪽)에만 붙이는 지시문
QUERY_INSTRUCTION = (
    "Instruct: Given a negative e-commerce review in Portuguese, "
    "retrieve reviews with the same cause of complaint\nQuery: "
)

# 경로는 실제 위치에 맞게 수정
pool_df = pl.read_csv(processed_dir / "pool.csv")
# utf-8-sig로 저장돼 첫 컬럼명에 BOM이 붙을 수 있어 제거
pool_df = pool_df.rename(lambda col_name: col_name.lstrip("\ufeff"))

# ── eval 불러오기 (pool_df는 앞에서 불러옴) ──
eval_df = pl.read_csv(processed_dir / "eval.csv")
eval_df = eval_df.rename(lambda col_name: col_name.lstrip("\ufeff"))

In [39]:
# ── 텍스트 목록 → 정규화된 임베딩 행렬 ──
def embed_texts(texts):
    embedding_rows = []
    # 한 번에 32건씩 묶어서 요청
    for batch_start in range(0, len(texts), EMBED_BATCH_SIZE):
        batch_texts = texts[batch_start : batch_start + EMBED_BATCH_SIZE]
        response = ollama.embed(model=EMBED_MODEL, input=batch_texts)
        embedding_rows.extend(response.embeddings)
    embedding_matrix = np.array(embedding_rows, dtype=np.float32)
    # 벡터 길이를 1로 맞춰서, 이후 내적(@)이 곧 코사인 유사도가 되게 함
    row_norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
    return embedding_matrix / row_norms

In [40]:
# ── 계산 & 저장 (review_id 순서도 함께 저장해 행 순서를 보장) ──
start_time = time.perf_counter()
pool_embeddings = embed_texts(pool_df["review_text"].to_list())
eval_embeddings = embed_texts(
    [QUERY_INSTRUCTION + review_text for review_text in eval_df["review_text"].to_list()]
)
print(pool_embeddings.shape, eval_embeddings.shape, f"{time.perf_counter() - start_time:.0f}초")

np.savez(
    EMBED_PATH,
    pool_review_ids=np.array(pool_df["review_id"].to_list(), dtype=str),
    pool_embeddings=pool_embeddings,
    eval_review_ids=np.array(eval_df["review_id"].to_list(), dtype=str),
    eval_embeddings=eval_embeddings,
)

(1000, 1024) (300, 1024) 68초


In [41]:
# ── 검증: pool 안에서 유사 리뷰 7개의 다수결 라벨이 자기 라벨과 맞는 비율 ──
# eval 라벨은 쓰지 않고, 임베딩이 "불만 원인"을 잘 담는지만 확인
pool_labels = np.array(pool_df["label"].to_list())
similarity_matrix = pool_embeddings @ pool_embeddings.T
np.fill_diagonal(similarity_matrix, -np.inf)  # 자기 자신은 이웃에서 제외
top_k_indices = np.argsort(-similarity_matrix, axis=1)[:, :7]

neighbor_labels = pool_labels[top_k_indices]
majority_labels = np.array([Counter(label_row).most_common(1)[0][0] for label_row in neighbor_labels])
knn_accuracy = (majority_labels == pool_labels).mean()

# 비교 기준: 전부 최빈 라벨(배송 지연)로 찍었을 때의 정확도
baseline_accuracy = (pool_labels == Counter(pool_labels).most_common(1)[0][0]).mean()
print(f"kNN 다수결 정확도: {knn_accuracy:.3f} / 최빈값 기준: {baseline_accuracy:.3f}")

kNN 다수결 정확도: 0.762 / 최빈값 기준: 0.351


# 프롬프트 예시 선택

In [42]:
# ── 라이브러리 ──
from pathlib import Path

# ── 설정 ──
K_EXAMPLES = 7
FIXED_SHUFFLE_SEED = 2026   # 고정 예시 순서 섞기용
RANDOM_BASE_SEED = 2026     # 랜덤 예시: 리뷰마다 (기본 시드 + eval 행 번호)
RESULTS_DIR = Path("../data/processed/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LABEL_KO_TO_CODE = {label_ko: label_code for label_code, label_ko in LABEL_CODE_TO_KO.items()}

# ── 저장된 임베딩 불러오기 + review_id 순서가 데이터프레임과 같은지 확인 ──
# (커널을 재시작해도 이 셀부터 이어서 실행 가능)
embedding_file = np.load(EMBED_PATH)
assert (embedding_file["pool_review_ids"] == np.array(pool_df["review_id"].to_list())).all()
assert (embedding_file["eval_review_ids"] == np.array(eval_df["review_id"].to_list())).all()
pool_embeddings = embedding_file["pool_embeddings"]
eval_embeddings = embedding_file["eval_embeddings"]

# 메시지 조립에 쓸 pool 텍스트·라벨 코드 목록
pool_texts = pool_df["review_text"].to_list()
pool_labels = np.array(pool_df["label"].to_list())
pool_codes = [LABEL_KO_TO_CODE[label_ko] for label_ko in pool_labels]

In [43]:
# ── 고정 예시: 라벨별 임베딩 평균(중심)에 가장 가까운 리뷰 1개씩 ──
def select_fixed_examples():
    fixed_indices = []
    for label_ko in LABEL_CODE_TO_KO.values():
        label_indices = np.where(pool_labels == label_ko)[0]
        centroid = pool_embeddings[label_indices].mean(axis=0)
        centroid = centroid / np.linalg.norm(centroid)
        closest_position = np.argmax(pool_embeddings[label_indices] @ centroid)
        fixed_indices.append(int(label_indices[closest_position]))
    # 라벨 순서가 그대로 드러나지 않게 시드로 한 번 섞어서 고정
    shuffle_rng = np.random.default_rng(FIXED_SHUFFLE_SEED)
    return shuffle_rng.permutation(fixed_indices).tolist()

FIXED_EXAMPLE_INDICES = select_fixed_examples()


# ── 랜덤 예시: 리뷰마다 7개, 리뷰별 시드 고정으로 재현 가능 ──
def select_random_examples(eval_position):
    example_rng = np.random.default_rng(RANDOM_BASE_SEED + eval_position)
    return example_rng.choice(len(pool_texts), size=K_EXAMPLES, replace=False).tolist()


# ── RAG 예시: 유사도 상위 7개, 가장 비슷한 예시가 마지막(리뷰 바로 앞) ──
def select_rag_examples(eval_position):
    similarities = pool_embeddings @ eval_embeddings[eval_position]
    top_indices = np.argsort(-similarities)[:K_EXAMPLES]
    return top_indices[::-1].tolist()


# ── 프롬프트 종류별로 예시 인덱스 반환 ──
def select_examples(prompt_type, eval_position):
    if prompt_type == "zero_shot":
        return []
    if prompt_type == "fixed":
        return FIXED_EXAMPLE_INDICES
    if prompt_type == "random":
        return select_random_examples(eval_position)
    if prompt_type == "rag":
        return select_rag_examples(eval_position)
    raise ValueError(f"알 수 없는 prompt_type: {prompt_type}")

In [44]:
# ── 메시지 조립: 시스템 프롬프트 → 예시 대화 턴 → 실제 리뷰 ──
def build_messages(example_indices, review_text):
    messages = [{"role": "system", "content": BASE_PROMPT}]
    for example_index in example_indices:
        messages.append({"role": "user", "content": pool_texts[example_index]})
        messages.append({"role": "assistant", "content": json.dumps({"label": pool_codes[example_index]})})
    messages.append({"role": "user", "content": review_text})
    return messages


# ── 분류 요청: 시스템 프롬프트 대신 조립된 메시지를 받도록 수정 ──
def classify_review(model_name, messages):
    start_time = time.perf_counter()
    response = ollama.chat(
        model=model_name,
        messages=messages,
        options={"temperature": 0},
        **MODEL_CONFIGS[model_name],
    )
    elapsed_sec = time.perf_counter() - start_time

    raw_output = response.message.content
    pred_code = parse_label_code(raw_output)
    return {
        "model": model_name,
        "pred_code": pred_code,
        "pred_label": LABEL_CODE_TO_KO.get(pred_code),
        "parse_ok": pred_code is not None,
        "raw_output": raw_output,
        "elapsed_sec": round(elapsed_sec, 2),
        "input_tokens": response.prompt_eval_count,
        "output_tokens": response.eval_count,
    }

In [45]:
# ── 점검 1: 고정 예시 7개가 "대표"로 적절한지 확인 ──
with pl.Config(fmt_str_lengths=150, tbl_rows=-1):
    print(pool_df[FIXED_EXAMPLE_INDICES].select("label", "review_text"))

# ── 점검 2: eval과 pool에 본문이 같은 리뷰 (대소문자·앞뒤 공백 무시) ──
normalized_pool_texts = pool_df["review_text"].str.to_lowercase().str.strip_chars()
duplicate_eval_df = eval_df.filter(
    pl.col("review_text").str.to_lowercase().str.strip_chars().is_in(normalized_pool_texts)
)
print(f"pool과 본문이 같은 eval 리뷰: {duplicate_eval_df.height}건")
with pl.Config(fmt_str_lengths=100, tbl_rows=-1):
    print(duplicate_eval_df.select("review_id", "review_text"))

# ── 점검 3: RAG 메시지가 의도대로 조립되는지 eval 첫 리뷰로 확인 ──
sample_messages = build_messages(select_examples("rag", 0), eval_df["review_text"][0])
for message in sample_messages[1:]:  # 시스템 프롬프트는 길어서 생략
    print(f'{message["role"]:9s} | {message["content"][:80]}')

shape: (7, 2)
┌─────────────┬────────────────────────────────────────────────────────────────────────────────────┐
│ label       ┆ review_text                                                                        │
│ ---         ┆ ---                                                                                │
│ str         ┆ str                                                                                │
╞═════════════╪════════════════════════════════════════════════════════════════════════════════════╡
│ 배송 지연   ┆ Ainda não recebi o produto...                                                      │
│ 제품 불량   ┆ Produto com defeito!                                                               │
│ 제품 불만족 ┆ produto de má qualidade                                                            │
│ 누락        ┆ Botei que não foi entregue, porque da compra que eu fiz de 4 produto da mesma nota │
│             ┆ só 1 veio.. estou esperando alguem entrar em contato                        

/var/folders/67/63m6y1m97g90zkz5c8wkzrdr0000gn/T/ipykernel_69120/2493489859.py:7: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  duplicate_eval_df = eval_df.filter(


In [46]:
# ── kNN 기준선: eval 리뷰마다 유사도 상위 7개 pool 라벨의 다수결 ──
# 동률이면 Counter가 먼저 나온 라벨(= 더 비슷한 이웃)을 고름
eval_top_indices = np.argsort(-(eval_embeddings @ pool_embeddings.T), axis=1)[:, :K_EXAMPLES]
knn_pred_labels = [Counter(pool_labels[top_row]).most_common(1)[0][0] for top_row in eval_top_indices]

knn_result_df = eval_df.select(
    "review_id", pl.col("label").alias("gold_label")
).with_columns(
    pl.lit("knn").alias("model"),
    pl.lit("knn_k7").alias("prompt_type"),
    pl.Series("pred_label", knn_pred_labels),
)
knn_result_df.write_parquet(RESULTS_DIR / "knn_k7.parquet")
print(knn_result_df.head())

shape: (5, 5)
┌─────────────────────────────────┬────────────┬───────┬─────────────┬────────────┐
│ review_id                       ┆ gold_label ┆ model ┆ prompt_type ┆ pred_label │
│ ---                             ┆ ---        ┆ ---   ┆ ---         ┆ ---        │
│ str                             ┆ str        ┆ str   ┆ str         ┆ str        │
╞═════════════════════════════════╪════════════╪═══════╪═════════════╪════════════╡
│ 23aafb33fe6fa8ccf9f9dd06b010b9… ┆ 기타       ┆ knn   ┆ knn_k7      ┆ 기타       │
│ 7150acdbbed2586e02ba3dead9271f… ┆ 배송 지연  ┆ knn   ┆ knn_k7      ┆ 배송 지연  │
│ 3517d6adbd335234632b206036b503… ┆ 배송 지연  ┆ knn   ┆ knn_k7      ┆ 배송 지연  │
│ cf27d64ca02457460c178a9781e48b… ┆ 누락       ┆ knn   ┆ knn_k7      ┆ 배송 지연  │
│ 779adfa09e476714bcc18fbf0ecb7e… ┆ 누락       ┆ knn   ┆ knn_k7      ┆ 누락       │
└─────────────────────────────────┴────────────┴───────┴─────────────┴────────────┘


# 오류 진단

- ResponseError: structured output is unavailable (status code: 501)

In [7]:
# ── 진단 설정: 출력 형식 3가지와 공통 프롬프트 ──
FORMAT_MODES = {
    "schema": OUTPUT_SCHEMA,  # JSON 스키마 강제
    "json": "json",           # JSON 형태만 강제
    "none": None,             # 강제 없음 (프롬프트 지시만)
}

# 세 방식 모두 같은 프롬프트를 써서 조건을 맞춤
DIAG_PROMPT = (
    "브라질 이커머스의 포르투갈어 불만 리뷰를 읽고, 불만의 주 원인을 "
    f"다음 라벨 중 하나로 분류하라: {', '.join(LABELS)}\n"
    '반드시 {"label": "라벨"} 형식의 JSON 한 줄로만 답하라.'
)

# ── 응답이 JSON으로 파싱되고 라벨이 7개 중 하나인지 검사 ──
def is_valid_label(raw_text):
    try:
        return json.loads(raw_text).get("label") in LABELS
    except (json.JSONDecodeError, AttributeError):
        return False

In [9]:
# ── 모델 × 형식 조합별로 호출해 결과 기록 ──
diag_records = []
for model_name, model_options in MODEL_CONFIGS.items():
    for mode_name, format_value in FORMAT_MODES.items():
        # 에러가 나도 표가 깨지지 않도록 모든 컬럼을 미리 채워둠
        record = {"model": model_name, "mode": mode_name, "status": None,
                  "valid": None, "has_thinking": None, "elapsed_sec": None, "output": None}
        try:
            start_time = time.perf_counter()
            response = ollama.chat(
                model=model_name,
                messages=[
                    {"role": "system", "content": DIAG_PROMPT},
                    {"role": "user", "content": sample_row["review_text"]},
                ],
                format=format_value,
                options={"temperature": 0},
                **model_options,
            )
            raw_text = response.message.content
            record.update(
                status="ok",
                valid=is_valid_label(raw_text),
                # 추론 텍스트가 별도 필드로 왔는지 (thinking이 실제로 꺼졌는지 확인용)
                has_thinking=bool(getattr(response.message, "thinking", None)),
                elapsed_sec=round(time.perf_counter() - start_time, 1),
                output=raw_text[:200],
            )
        except ollama.ResponseError as error:
            record.update(status=f"error {error.status_code}", output=str(error.error)[:200])
        diag_records.append(record)

# ── 결과 표 출력 (긴 문자열도 잘리지 않게) ──
diag_df = pl.DataFrame(diag_records)
with pl.Config(fmt_str_lengths=200, tbl_rows=-1):
    print(diag_df)

shape: (6, 7)
┌───────────────────┬────────┬───────────┬───────┬──────────────┬─────────────┬────────────────────┐
│ model             ┆ mode   ┆ status    ┆ valid ┆ has_thinking ┆ elapsed_sec ┆ output             │
│ ---               ┆ ---    ┆ ---       ┆ ---   ┆ ---          ┆ ---         ┆ ---                │
│ str               ┆ str    ┆ str       ┆ bool  ┆ bool         ┆ f64         ┆ str                │
╞═══════════════════╪════════╪═══════════╪═══════╪══════════════╪═════════════╪════════════════════╡
│ qwen3.5:4b        ┆ schema ┆ error 501 ┆ null  ┆ null         ┆ null        ┆ structured output  │
│                   ┆        ┆           ┆       ┆              ┆             ┆ is unavailable     │
│ qwen3.5:4b        ┆ json   ┆ error 501 ┆ null  ┆ null         ┆ null        ┆ structured output  │
│                   ┆        ┆           ┆       ┆              ┆             ┆ is unavailable     │
│ qwen3.5:4b        ┆ none   ┆ ok        ┆ true  ┆ false        ┆ 1.9        